In [9]:
import sys, importlib
MODULE_DIR = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/script"
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import pro_assoc_tools
importlib.reload(pro_assoc_tools)

<module 'pro_assoc_tools' from '/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/script/pro_assoc_tools.py'>

In [10]:
pro_ex_path = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/Proteome/Soma_QC_re/Results/03.phom_day0/PHOM.ex.day0.csv"
pro_apt_path = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/Proteome/Soma_QC_re/Results/03.phom_day0/PHOM.apt.day0.csv"
plink_prefix = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/wgs/19.tommo_panel_filter/cteph_agp3k.lowfreq_common"
gwas_path = "/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/analysis/assoc_plink2/results/04.summary_vis_snp/gwas_summary.plink2.csv"

In [11]:
# 提取TEST列包含'ADD'的ID形成list
import ast
import pandas as pd

gwas_df = pd.read_csv(gwas_path)

# 解析字符串为列表后检查
def contains_add(test_str):
    try:
        test_list = ast.literal_eval(test_str)
        return 'ADD' in test_list
    except:
        return False

variant_ids_method = gwas_df[gwas_df['TEST'].apply(contains_add)]['ID'].tolist()
variant_ids = variant_ids_method

print(f"包含'ADD'的变异ID数量: {len(variant_ids)}")
print(f"变异ID列表: {variant_ids}")

包含'ADD'的变异ID数量: 5
变异ID列表: ['chr3:154069965:A:G', 'chr16:53884344:G:A', 'chr16:53887925:T:C', 'chr17:13528059:G:A', 'chr17:13528180:G:A']


In [12]:
import pandas as pd
import re

# 读取蛋白质表达数据文件
pro_ex_df = pd.read_csv(pro_ex_path, index_col=0)

# 提取行名（样本ID）
sample_ids_raw = pro_ex_df.index.tolist()

# 去掉_day*形式的后缀
sample_ids = []
for sample_id in sample_ids_raw:
    # 使用正则表达式去掉_day后面跟数字的部分
    clean_id = re.sub(r'_day\d*$', '', str(sample_id))
    sample_ids.append(clean_id)

print(f"原始样本ID数量: {len(sample_ids_raw)}")
print(f"处理后样本ID数量: {len(sample_ids)}")
print(f"前10个原始样本ID: {sample_ids_raw[:10]}")
print(f"前10个处理后样本ID: {sample_ids[:10]}")

原始样本ID数量: 43
处理后样本ID数量: 43
前10个原始样本ID: ['PHOM0126_day0', 'PHOM0108_day0', 'PHOM0009_day0', 'PHOM0099_day0', 'PHOM0129_day0', 'PHOM0128_day0', 'PHOM0080_day0', 'PHOM0112_day0', 'PHOM0064_day0', 'PHOM0091_day0']
前10个处理后样本ID: ['PHOM0126', 'PHOM0108', 'PHOM0009', 'PHOM0099', 'PHOM0129', 'PHOM0128', 'PHOM0080', 'PHOM0112', 'PHOM0064', 'PHOM0091']


In [13]:
from pro_assoc_tools import export_plink_geno_matrix

gt_mx = export_plink_geno_matrix(
    plink_prefix=plink_prefix,
    output_tsv="gt_mx.tsv",
    variant_ids=variant_ids,
    sample_ids=sample_ids,
    plink_threads=8
)

[INFO] 输出：gt_mx.tsv
[INFO] 工作目录：/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc；缓存目录：/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/tmp_export
[INFO] 读取 FAM：样本总数=3017
[INFO] BIM ID 风格：chr*
[INFO] BIM 变体总数=5677100
[INFO] 执行命令（fallback）: awk -F'\t' '{split($2,a,":"); if (length(a)>=4) print $2"\t"a[4]; else {print "[ERR] Bad BIM ID: "$2 > "/dev/stderr"; exit 1}}' /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/wgs/19.tommo_panel_filter/cteph_agp3k.lowfreq_common.bim > /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/tmp_export/export_allele.from_bim_id_ALT.txt
[INFO] [变体选择] 请求=5；按位点命中请求=5（100.00%）；BIM提取ID数=5
[INFO] --extract 使用 5 个 BIM ID（由 5 个请求位点映射而来）
[INFO] [样本选择] 请求=43；命中=43（100.00%）；未命中=0；FAM总数=3017
[INFO] --keep 使用 43 个样本
[INFO] [导出计划] 变体：5/5；样本：43/43；线程：plink=8
[INFO] 执行命令（fallback）: /home/b/b37974/plink2 --bfile /LARGE0/gr10478/b37974/Pulmonary_Hypertensi

In [14]:
from pro_assoc_tools import summarize_gt_counts

summary_path = summarize_gt_counts(
    gt_tsv_path=gt_mx,
    out_path="gt_counts_summary.tsv",
    preview_n=10,
    print_preview=True
)

[INFO] [GT-SUM] 输入矩阵：gt_mx.tsv
[INFO] [GT-SUM] 输出汇总：gt_counts_summary.tsv
[INFO] [GT-SUM] 完成，累计变体数=5 行；每行含 n0/n1/n2/nmiss/ncalled/nsamples。
[INFO] [GT-SUM] 预览前 5 行：
ID	n0	n1	n2	nmiss	ncalled	nsamples
chr3:154069965:A:G	38	3	0	2	41	43
chr16:53884344:G:A	23	15	3	2	41	43
chr16:53887925:T:C	25	15	3	0	43	43
chr17:13528059:G:A	35	7	0	1	42	43
chr17:13528180:G:A	34	7	0	2	41	43


In [15]:
from pro_assoc_tools import assemble_variant_gene_protein_table

variant_meta = assemble_variant_gene_protein_table(
    variant_ids=variant_ids,
    gwas_path=gwas_path,
    pro_apt_path=pro_apt_path,
    out_path="variant_meta.tsv",
    preview_n=10,
    print_preview=True
)

[INFO] [AGG] 读取 GWAS：/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/analysis/assoc_plink2/results/04.summary_vis_snp/gwas_summary.plink2.csv
[INFO] [AGG] 读取 pro_apt：/LARGE0/gr10478/b37974/Pulmonary_Hypertension/Proteome/Soma_QC_re/Results/03.phom_day0/PHOM.apt.day0.csv
[INFO] [AGG] 已写出：variant_meta.tsv（5 行）
[INFO] [AGG] 预览前 5 行：
ID	rsID	Gene	SeqID	UniProt	Warn
chr3:154069965:A:G	rs1021580972	ARHGEF26-AS1	na	na	no_pro_apt_for_gene:ARHGEF26-AS1
chr16:53884344:G:A	rs56278663	FTO	X25051-104	Q9C0B1	
chr16:53887925:T:C	rs16952623	FTO	X25051-104	Q9C0B1	
chr17:13528059:G:A	rs34804183	HS3ST3A1	X8268-98	Q9Y663	
chr17:13528180:G:A	rs34480904	HS3ST3A1	X8268-98	Q9Y663


In [16]:
from pro_assoc_tools import plot_protein_boxplot_per_variant

models = ["ADD", "DOM", "REC"]
out_pdfs = {}

for m in models:
    out_pdfs[m] = plot_protein_boxplot_per_variant(
        variant_meta=variant_meta,
        summary_path=summary_path,
        pro_ex_path=pro_ex_path,
        gt_mx=gt_mx,
        model=m,
        log2_transform=True,   # 保持你原来的设置
        # expr_is_log2=None,   # 若你显式知道已是log2，可设为True；否则留None自动判断
    )
    print(f"[{m}] PDF at:", out_pdfs[m])

print("All PDFs:", out_pdfs)


[INFO] [PLOT] 日志写出：/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.ADD.pdf.log，共 8 条记录
[INFO] [PLOT] PDF 生成完成：/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.ADD.pdf


[ADD] PDF at: /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.ADD.pdf


[INFO] [PLOT] 日志写出：/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.DOM.pdf.log，共 8 条记录
[INFO] [PLOT] PDF 生成完成：/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.DOM.pdf


[DOM] PDF at: /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.DOM.pdf


[INFO] [PLOT] 日志写出：/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.REC.pdf.log，共 8 条记录
[INFO] [PLOT] PDF 生成完成：/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.REC.pdf


[REC] PDF at: /LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.REC.pdf
All PDFs: {'ADD': '/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.ADD.pdf', 'DOM': '/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.DOM.pdf', 'REC': '/LARGE0/gr10478/b37974/Pulmonary_Hypertension/cteph_agp3k/review_analysis/01.pro_assoc/protein_genotype_boxplots.REC.pdf'}
